# Food Recognition & 3D Plate Scanning Research Pipeline
### Google Colab GPU Backup Training & Dataset Acquisition Pipeline (T4 / V100 / A100)

**Source of Truth Repository:** `https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git`  
**Primary Platform:** Kaggle GPU is primary. Use this Colab notebook as a secondary/backup compute environment.

--- 
### ⚠️ Colab Persistence & Compliance Notice
1. **Ephemeral Local Storage:** `/content` is destroyed when the Colab runtime disconnects. To ensure permanent checkpoint persistence, optional Google Drive mounting is provided below.
2. **Strict License Compliance:** Only datasets approved for the commercial workflow are downloaded. Non-commercial sources (e.g. Khana CC BY-NC-ND) are strictly quarantined.
3. **Zero Hardcoded Secrets:** Never paste GitHub tokens or credentials into public cells.

### Step 1: Clone GitHub Repository (Source of Truth)

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git"
REPO_DIR = "/content/food-plat-scanner-ai-model"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest commits...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    print(f"Cloning source of truth repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current Working Directory: {os.getcwd()}")

### Step 2: Hardware, GPU & Disk Storage Environment Audit

In [ ]:
import shutil
import urllib.request
import torch

print("=" * 85)
print("GOOGLE COLAB RUNTIME & HARDWARE AUDIT")
print("=" * 85)
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Accelerator:  {device_name}")
    print(f"Total Dedicated VRAM:    {total_mem_gb:.2f} GB")
else:
    print("WARNING: GPU is not active! Navigate to Runtime -> Change runtime type -> T4 GPU.")

total_disk, used_disk, free_disk = shutil.disk_usage("/content")
print(f"Disk Space (/content): Total={total_disk/(1024**3):.1f} GB | Free={free_disk/(1024**3):.1f} GB")
print("=" * 85)

### Step 3 (Optional): Mount Google Drive for Persistent Storage

In [ ]:
# Mount Google Drive if persistent cloud saving is desired
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_EXPORT_DIR = "/content/drive/MyDrive/food_model_checkpoints"
    os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
    print(f"Google Drive mounted successfully. Checkpoints will sync to: {DRIVE_EXPORT_DIR}")
except Exception as e:
    print(f"Skipping Google Drive mount (or running outside Colab UI): {e}")

### Step 4: Install Required Dependencies

In [ ]:
!pip install -q pyarrow fastparquet Pillow torchvision
print("Dependencies installed successfully.")

### Step 5: Run Scientific Unit & Integrity Tests

In [ ]:
!python -m src.utils.run_unit_tests

### Step 6: Run Resumable Downloader for Approved Datasets
Acquires the approved 15,404-image Enhanced Indian Food dataset directly into `/content/datasets/raw/`.

In [ ]:
from src.data.resumable_downloader import CloudDatasetOrchestrator, EnhancedIndianFoodAdapter

orchestrator = CloudDatasetOrchestrator()
plan = orchestrator.generate_acquisition_plan()

adapter = EnhancedIndianFoodAdapter(staging_dir="/content/datasets/raw/sohlhealth_indian_food")
success = adapter.acquire(resume=True)
print(f"Dataset acquisition status: {'SUCCESS' if success else 'FAILED'}")

### Step 7: Run Smoke Test on Pretrained MobileNetV3-Small

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(in_features, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.2),
    nn.Linear(1024, 10)
)
model = model.to(device)

dummy_input = torch.randn(4, 3, 224, 224, device=device)
output = model(dummy_input)
print(f"Smoke Test Output Shape: {output.shape} (Expected: [4, 10])")
print("Smoke test verified on Colab hardware.")

### Step 8: Execute Staged Transfer Learning Training

In [ ]:
!python -m src.classification.train_kaggle

### Step 9: Package & Persist Artifacts

In [ ]:
import shutil
import zipfile

zip_dest = "/content/colab_run_artifacts.zip"
with zipfile.ZipFile(zip_dest, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk("outputs"):
        for f in files:
            zipf.write(os.path.join(root, f), arcname=os.path.join("outputs", f))
    for root, _, files in os.walk("models"):
        for f in files:
            if f.endswith(".pth"):
                zipf.write(os.path.join(root, f), arcname=os.path.join("models", f))

print(f"Artifact bundle created at {zip_dest} ({os.path.getsize(zip_dest)/(1024**2):.2f} MB).")
if os.path.exists("/content/drive/MyDrive/food_model_checkpoints"):
    shutil.copy(zip_dest, "/content/drive/MyDrive/food_model_checkpoints/")
    print("Artifact bundle synced to persistent Google Drive storage.")